# LLaMA-2 + LoRA + 8-bit 量化指令微調（2026 版）

## 學習目標

1. 理解 8-bit 量化與 4-bit 量化的差異，以及何時選擇 8-bit。
2. 掌握 2026 正確的量化載入流程：`BitsAndBytesConfig` → `prepare_model_for_kbit_training()` → `get_peft_model()`。
3. 用 `trl.SFTTrainer` 取代手刻 `-100` 標籤的 `Trainer`，同時保留底層手刻版作對照理解。
4. 用 `tokenizer.apply_chat_template()` 建立訓練/推論一致的對話模板。
5. 能夠在訓練結束後把 LoRA adapter merge 回 base model，並推送至 Hugging Face Hub。

## 前置知識

- 已完成 `04-kbits-tuning/01-4bits_training/` — 了解 4-bit QLoRA 基本流程
- 已完成 `04-kbits-tuning/02-qlora/` — 了解 PEFT LoRA 設定
- 基礎 PyTorch / Transformers 使用經驗

## 銜接

- 上一個 notebook：`../02-qlora/qlora_llama2.ipynb`（4-bit QLoRA）
- 下一個 notebook：`../../05-Multimodal/`（AutoProcessor 多模態訓練）

In [ ]:
# 版本鎖定 — 確保可重現性
# 建議在新環境執行：pip install -r requirements.txt
# 最低需求如下（可直接執行此 cell 安裝）

import subprocess, sys

PACKAGES = [
    "transformers>=4.46",
    "datasets>=3.0",
    "trl>=0.12",
    "peft>=0.13",
    "accelerate>=1.0",
    "bitsandbytes>=0.44",
    "evaluate>=0.4",
    "safetensors>=0.4",
    "torch>=2.4",
]

# 僅在版本不符時安裝，避免每次重跑浪費時間
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("All packages up to date.")

## Step 1：匯入套件

2026 主要套件說明：
- `BitsAndBytesConfig`：量化設定物件，傳給 `quantization_config=` 參數（transformers 4.42+）
- `prepare_model_for_kbit_training`：在 PEFT 前必須呼叫，用途見 Step 4
- `SFTTrainer` / `SFTConfig`：`trl` 提供的指令微調 trainer，自動處理 response-only 標籤遮罩
- `set_seed`：確保訓練可重現

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# 全局隨機種子 — 讓訓練結果可重現
set_seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2：載入資料集

從 Hugging Face Hub 載入資料集，確保任何環境都可重現。
若你有自訂資料集，可用 `datasets.Dataset.from_pandas()` 或 `load_dataset('json', data_files=...)` 替換。

In [ ]:
# Alpaca 中文資料集 — HF Hub 公開版
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
print(f"Dataset size: {len(ds):,}")
print(f"Columns     : {ds.column_names}")

In [ ]:
# 查看前 3 筆資料，確認欄位結構
for i in range(3):
    print(f"--- example {i} ---")
    print("instruction:", ds[i]["instruction"][:80])
    print("input      :", ds[i].get("input", "")[:80])
    print("output     :", ds[i]["output"][:80])
    print()

## Step 3：初始化 Tokenizer

### 為什麼 `padding_side='right'`？

Decoder-only 模型（LLaMA 系列）訓練時若 batch > 1，padding 必須放在右側（序列尾端）。
若放左側，attention mask 的 causal 遮罩會讓 padding token 被當作有效 prefix，導致梯度不收斂。

### `pad_token_id = 2`

LLaMA-2 原始 tokenizer 沒有設定 `pad_token`（只有 `eos_token` = `</s>` = id 2）。
訓練時需要指定 pad token，常見做法是複用 eos token 的 id，但注意訓練資料的 labels 中不要把 pad 位置算進 loss（DataCollatorForSeq2Seq 或 SFTTrainer 會自動處理）。

In [ ]:
# model_id 使用 HF Hub id
# 若網路受限，可設 HF_HUB_OFFLINE=1 並指向本機快取
MODEL_ID = "meta-llama/Llama-2-7b-hf"
# 輕量替代（< 8 GB VRAM）：
# MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Decoder-only 模型 batch 訓練必須右側 padding，否則 batch > 1 時容易不收斂
tokenizer.padding_side = "right"

# LLaMA-2 tokenizer 沒有獨立 pad token，慣例複用 eos token id = 2
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id  # id = 2

print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"EOS token id   : {tokenizer.eos_token_id}")
print(f"Pad token id   : {tokenizer.pad_token_id}")
print(f"Padding side   : {tokenizer.padding_side}")

## Step 3b：對話模板 — `apply_chat_template()`

每個現代模型的 tokenizer 都內建 Jinja2 模板，記錄了該模型訓練時使用的對話格式。
`apply_chat_template()` 讀取這個模板，自動產生正確格式——換模型只需換 `MODEL_ID`，不需改任何字串。

**訓練/推論一致性**：訓練時和推論時用同一套模板，確保分布匹配，這是避免「訓練推論不一致」問題的根本解法。

In [ ]:
# 示範 apply_chat_template 的效果
example_messages = [
    {"role": "user", "content": "你好，請介紹一下自己"},
    {"role": "assistant", "content": "你好！我是一個 AI 助手，很高興為你服務。"},
]

chat_str = tokenizer.apply_chat_template(
    example_messages,
    tokenize=False,
    add_generation_prompt=False,  # 推論時設 True，訓練時設 False（不加尾部 prompt prefix）
)
print("--- apply_chat_template 輸出 ---")
print(repr(chat_str))
print()
print("--- 可讀形式 ---")
print(chat_str)

## Step 3c：資料預處理 — 底層手刻版與 SFTTrainer 路徑

本節保留底層手刻版，讓你理解 `-100` 遮罩的原理；
預設訓練路徑使用 `SFTTrainer` 的 `formatting_func`，由 trl 自動處理標籤遮罩。

### 底層手刻版（理解用）

為什麼 `labels` 要把 instruction 部分設成 `-100`？

PyTorch 的 `CrossEntropyLoss` 預設忽略 target == `-100` 的位置，不計算該位置的 loss。
指令微調只希望模型學會「如何回應」，不需要學習「如何重複 instruction」，
因此把 instruction token 的 label 設為 `-100`，讓 loss 只從 response token 開始計算。

In [ ]:
# ====== 底層手刻版（理解 -100 遮罩原理用）======
# 此版本保留作教學對照，實際訓練路徑見 Step 6 的 SFTTrainer

MAX_LENGTH = 384  # LLaMA tokenizer 會把一個中文字切成多個 token，需要稍大的長度

def build_messages(example: dict) -> list[dict]:
    """Convert alpaca-style example to chat messages list."""
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = user_content + "\n" + example["input"]
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]

def process_func_manual(example: dict) -> dict:
    """Manually build input_ids / attention_mask / labels with -100 masking.

    This is the LOW-LEVEL reference implementation.
    SFTTrainer with formatting_func does the equivalent automatically.
    """
    messages = build_messages(example)

    # 只取 instruction 部分（不含 assistant turn），加 add_generation_prompt=True 讓 tokenizer 加上 response 的起始 token
    instruction_str = tokenizer.apply_chat_template(
        messages[:-1],  # 只取 user turn
        tokenize=False,
        add_generation_prompt=True,
    )
    response_str = example["output"] + tokenizer.eos_token

    instruction_ids = tokenizer(instruction_str, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_str, add_special_tokens=False)["input_ids"]

    input_ids = instruction_ids + response_ids
    attention_mask = [1] * len(input_ids)
    # instruction 部分 label 設 -100，不計入 loss
    labels = [-100] * len(instruction_ids) + response_ids

    # 截斷到 MAX_LENGTH
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# 只用少量資料驗證格式
tokenized_demo = ds.select(range(5)).map(process_func_manual, remove_columns=ds.column_names)
print("Manual tokenized sample:")
print(f"  input_ids length : {len(tokenized_demo[0]['input_ids'])}")
print(f"  label -100 count : {tokenized_demo[0]['labels'].count(-100)}")
print()
# 解碼驗證：labels 中 -100 以外的部分應該只有 response
visible_labels = [x for x in tokenized_demo[0]["labels"] if x != -100]
print("Response tokens decoded:")
print(tokenizer.decode(visible_labels))

## Step 4：載入模型 — 8-bit 量化

### 為何選 8-bit 而非 4-bit？

| 精度 | VRAM 需求（7B）| 訓練速度 | 精度損失 | 適用場景 |
|------|--------------|----------|----------|----------|
| fp16/bf16 | ~14 GB | 快 | 無 | 有足夠 VRAM |
| 8-bit (LLM.int8) | ~8 GB | 中等 | 極小 | 需要最佳推論精度、VRAM 中等 |
| 4-bit NF4 | ~5 GB | 稍慢 | 小 | VRAM 最少、研究/實驗 |

8-bit 使用 bitsandbytes 的 LLM.int8() 演算法：matrix multiplication 時動態解量化，在 fp16 下計算，計算完再量化回 int8。精度損失比 4-bit 更小，適合需要更高輸出品質的場景。

### 為何 `bfloat16` 優於 `float16`？

- **數值範圍**：bf16 的指數位元數與 fp32 相同（8 bits），動態範圍更大，訓練時梯度 overflow/underflow 風險更低。
- **fp16 的缺點**：動態範圍小，梯度爆炸更常見，通常需要 `loss_scale` 補救；量化計算的 compute dtype 用 fp16 容易在某些 GPU 架構上出現 nan。
- **實測**：A100/H100 的 bf16 throughput 與 fp16 相同；RTX 30/40 系列 bf16 速度略慢但數值穩定性更好。

### `BitsAndBytesConfig` 量化設定

使用 `BitsAndBytesConfig` 物件並傳給 `quantization_config=`，讓量化設定明確、可序列化（存進 `config.json`），程式碼意圖清晰。

### `prepare_model_for_kbit_training()` 的正確順序

量化模型載入後，某些層（LayerNorm、lm_head）需要保持在較高精度以穩定訓練。
`prepare_model_for_kbit_training()` 會：
1. 把 LayerNorm 的 weight 轉為 fp32。
2. 凍結 base model 的所有參數（量化權重不訓練）。
3. 開啟 gradient checkpointing（節省 activation memory）。

**必須在 `get_peft_model()` 之前呼叫**，否則 LoRA adapter 的梯度計算可能失敗。

In [ ]:
# 1. 建立量化設定物件
# VRAM 預估：LLaMA-2-7B in 8-bit ~= 8 GB
# 若 VRAM < 8 GB，改用 load_in_4bit=True（見 ../01-4bits_training/）
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    # 8-bit 推論時的 compute dtype 仍建議用 bf16（比 fp16 數值穩定）
    # bitsandbytes >= 0.43 才支援此參數，舊版可省略
    llm_int8_threshold=6.0,          # outlier 閾值，預設 6.0 通常最佳
    llm_int8_has_fp16_weight=False,  # True 會保留 fp16 副本（更快但更耗記憶體）
)

# 2. 載入模型
# device_map='auto': transformers 自動分配 GPU/CPU/disk
# torch_dtype=torch.bfloat16: 非量化層（LayerNorm, lm_head）使用 bf16
# use_safetensors=True: 安全格式，避免 pickle arbitrary code execution 風險，載入更快
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)

print("Model loaded. Device map:")
print(model.hf_device_map)

In [ ]:
# 查看各層的 dtype，確認 int8 量化生效
for name, param in model.named_parameters():
    if "weight" in name:
        print(f"{name:60s}  shape={str(param.shape):20s}  dtype={param.dtype}")
        break  # 只印第一層示範，完整版可移除 break

In [ ]:
# 3. 量化後 PEFT 前置步驟（重要！）
# 必須在 get_peft_model() 之前呼叫
# - 將 LayerNorm 轉為 fp32（數值穩定性）
# - 凍結 base model 參數（只訓練 LoRA adapter）
# - 開啟 gradient checkpointing（節省 activation VRAM）
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("prepare_model_for_kbit_training() done.")

## Step 5：設定 LoRA Adapter

### LoRA 超參數說明

| 參數 | 說明 | 本範例值 |
|------|------|----------|
| `r` | LoRA rank，adapter 的低秩矩陣維度。越大可學習越多，但參數也越多 | 8 |
| `lora_alpha` | 縮放因子，等效 learning rate 縮放。慣例設為 `2 * r` | 16 |
| `lora_dropout` | Dropout 防止 adapter 過擬合 | 0.05 |
| `target_modules` | 要插入 LoRA 的層名稱。`q_proj`/`v_proj` 是最常見的選擇 | q_proj, v_proj |
| `bias` | 是否也訓練 bias。`none` 最省 VRAM | none |

### 為何只 target `q_proj` 和 `v_proj`？

Attention 的 Q 和 V 矩陣對模型行為影響最大（Q 決定「問什麼」，V 決定「取什麼值」）。
若 VRAM 充裕，可加入 `k_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` 以提升品質。

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    inference_mode=False,  # 訓練模式
)

# get_peft_model 必須在 prepare_model_for_kbit_training 之後呼叫
model = get_peft_model(model, lora_config)

# 開啟 input require grads（gradient checkpointing 時必須）
model.enable_input_require_grads()

# 查看可訓練參數數量
model.print_trainable_parameters()
# 預期輸出範例：trainable params: 4,194,304 || all params: 3,504,230,400 || trainable%: 0.12

## Step 6：訓練 — SFTTrainer

`SFTTrainer` 是 trl 提供的 Supervised Fine-Tuning 專用 trainer，自動處理：
- **Response-only loss masking**：透過 `DataCollatorForCompletionOnlyLM` 自動把 instruction 部分的 label 設為 `-100`，不需手刻。
- **Chat template 整合**：直接接受 `messages` 欄位或 `formatting_func`，自動套用模板。
- **Sequence packing**：多個短樣本打包成一個 sequence 提升 GPU 使用率（需設 `packing=True`）。

### `SFTConfig` 參數解說

| 參數 | 說明 |
|------|------|
| `bf16=True` | 非量化層使用 bf16 混合精度，比 fp16 更穩定 |
| `warmup_ratio=0.1` | 訓練前 10% steps 線性 warmup，防止初始 loss spike |
| `lr_scheduler_type='cosine'` | Cosine decay 比 linear 更平滑，通常收斂更好 |
| `gradient_accumulation_steps=32` | 有效 batch = 1 × 32 = 32，等效較大 batch 但不增加 VRAM |
| `max_grad_norm=1.0` | 梯度裁剪，防止梯度爆炸 |
| `save_safetensors=True` | 存成 safetensors 格式（安全、快速）|
| `optim='adamw_torch_fused'` | Fused AdamW：比標準 AdamW 快 ~10%（PyTorch 2.0+ 支援）|
| `seed=42` | 訓練可重現 |

In [ ]:
# formatting_func：將 alpaca 資料轉為 chat template 格式字串
# SFTTrainer 會將這個字串 tokenize，並自動遮罩 instruction 部分的 label
def formatting_func(example: dict) -> str:
    """Convert one alpaca-style example to a formatted chat string."""
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = user_content + "\n" + example["input"]
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


# 驗證 formatting_func 輸出
print(formatting_func(ds[0]))

In [ ]:
# DataCollatorForCompletionOnlyLM：自動遮罩 instruction 部分，只計算 response 的 loss
# response_template 是模型回應起始的 token 字串（與 chat template 的格式一致）
# LLaMA-2 的 chat template 在 assistant 回應前加 "[/INST] "
response_template = "[/INST]"
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

print(f"Response template tokens: {tokenizer.encode(response_template, add_special_tokens=False)}")

In [ ]:
# 使用前 6000 筆，與原始 notebook 一致
train_ds = ds.select(range(6000))

sft_config = SFTConfig(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,    # 有效 batch = 1 × 32 = 32
    num_train_epochs=1,
    logging_steps=10,
    save_steps=200,
    # --- 穩定訓練設定 ---
    bf16=True,                          # 混合精度（非量化層）
    warmup_ratio=0.1,                   # 訓練前 10% steps 做 warmup
    lr_scheduler_type="cosine",         # cosine decay
    learning_rate=2e-4,                 # LoRA 訓練常用 learning rate
    max_grad_norm=1.0,                  # 梯度裁剪
    optim="adamw_torch_fused",          # Fused AdamW（PyTorch 2.0+ 支援）
    save_safetensors=True,              # 存成 safetensors 格式
    gradient_checkpointing=True,        # 用時間換 VRAM（8-bit 量化時幾乎必須）
    seed=42,
    # SFT 特有設定
    max_seq_length=384,                 # 對應原本的 MAX_LENGTH
    packing=False,                      # 不打包序列（保持與原始 notebook 一致）
    dataset_text_field=None,            # 使用 formatting_func 時設為 None
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    formatting_func=formatting_func,
    data_collator=data_collator,
    processing_class=tokenizer,
    peft_config=lora_config,            # SFTTrainer 也可在這裡接受 peft_config（若 model 尚未套用）
)

print("SFTTrainer created.")
print(f"Training steps per epoch : {len(trainer.get_train_dataloader())}")
print(f"Effective batch size      : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")

## Step 7：執行訓練

### VRAM 估算（LLaMA-2-7B + 8-bit LoRA）

| 元素 | 大小估算 |
|------|----------|
| Base model (int8) | ~8 GB |
| LoRA adapter (bf16) | ~50 MB |
| Activations + gradient checkpointing | ~2-3 GB |
| Optimizer state (AdamW, bf16 master) | ~200 MB |
| **合計** | **~11 GB** |

若 VRAM < 11 GB，可降低 `max_seq_length` 至 256 或改用 4-bit（見 `../01-4bits_training/`）。

In [ ]:
trainer.train()

## Step 8：儲存 LoRA Adapter

訓練完成後先存 adapter（只有幾十 MB），再決定是否 merge 成完整模型。

In [ ]:
# 儲存 LoRA adapter
# safe_serialization=True：存成 safetensors 格式（安全、快速載入）
adapter_save_path = "./chatbot/lora_adapter"
model.save_pretrained(adapter_save_path, safe_serialization=True)
tokenizer.save_pretrained(adapter_save_path)
print(f"LoRA adapter saved to: {adapter_save_path}")

## Step 9：模型推理

### 推論時使用 `apply_chat_template(add_generation_prompt=True)`

推論時需要告訴模型「現在該生成 assistant 回應了」，因此設 `add_generation_prompt=True`，
這會在 prompt 尾端加上 assistant 的起始 token（與 chat template 中定義的格式一致）。

訓練時設 `add_generation_prompt=False`（因為標籤資料包含完整的 assistant 回應）。

In [ ]:
model.eval()

test_messages = [
    {"role": "user", "content": "你好，請用三句話介紹台灣"},
]

prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,  # 推論時設 True
)

ipt = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **ipt,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

# 只解碼新生成的部分（去掉 prompt）
new_tokens = output_ids[0, ipt["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("=== Model response ===")
print(response)

## Step 10：Merge LoRA → Base Model（可選）

`merge_and_unload()` 將 LoRA 的低秩矩陣 `B×A×alpha/r` 加回原始權重，
產生一個純 dense 模型（不再需要 PEFT），可獨立部署，不依賴 peft library。

**注意**：merge 後的模型恢復為原始精度（bf16），不再是 int8；
若要維持量化狀態部署，可直接使用 adapter（不 merge）。

In [ ]:
# merge LoRA adapter 回 base model
# merge_and_unload() 回傳的是 base model 類別（PeftModel 的封裝移除）
merged_model = model.merge_and_unload()
print(f"Merged model type: {type(merged_model).__name__}")

# 儲存 merged model
merged_save_path = "./chatbot/merged_model"
merged_model.save_pretrained(
    merged_save_path,
    safe_serialization=True,   # 存成 safetensors，取代 pickle 的 .bin 格式
)
tokenizer.save_pretrained(merged_save_path)
print(f"Merged model saved to: {merged_save_path}")

## Step 11：推送至 Hugging Face Hub（可選）

推送前需執行 `huggingface-cli login` 或設定 `HF_TOKEN` 環境變數。

推送時附上最小 model card（語言、授權、任務 tag）讓模型可被搜尋與引用。

In [ ]:
import os

# HF_TOKEN 從環境變數讀取，不要硬寫在程式碼裡
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    HUB_REPO_ID = "your-username/llama2-7b-lora-alpaca-zh"  # 替換為你的 Hub repo

    # 只推送 adapter（輕量，幾十 MB）
    model.push_to_hub(
        HUB_REPO_ID,
        token=HF_TOKEN,
        commit_message="Upload LLaMA-2-7B LoRA adapter (8-bit training)",
    )
    tokenizer.push_to_hub(HUB_REPO_ID, token=HF_TOKEN)

    # 最小 model card（必填欄位）
    from huggingface_hub import ModelCard
    card_content = f"""---
language:
  - zh
license: llama2
tags:
  - llama-2
  - lora
  - instruction-tuning
  - chinese
base_model: {MODEL_ID}
---
# LLaMA-2-7B LoRA Adapter — Alpaca ZH

This adapter was fine-tuned on the Chinese Alpaca dataset using 8-bit LoRA.

## Training details
- Base model: `{MODEL_ID}`
- Quantization: 8-bit (bitsandbytes LLM.int8)
- LoRA rank: 8, alpha: 16
- Target modules: q_proj, v_proj
- Training data: silk-road/alpaca-data-gpt4-chinese (6,000 samples)
"""
    card = ModelCard(card_content)
    card.push_to_hub(HUB_REPO_ID, token=HF_TOKEN)
    print(f"Pushed to https://huggingface.co/{HUB_REPO_ID}")
else:
    print("HF_TOKEN not set. Skipping push_to_hub.")
    print("To push: export HF_TOKEN=hf_...")

## 小結

### 本 notebook 核心流程

```
BitsAndBytesConfig(load_in_8bit=True)
        ↓
AutoModelForCausalLM.from_pretrained(device_map='auto', torch_dtype=bfloat16)
        ↓
prepare_model_for_kbit_training()        ← 量化後 PEFT 前的必要步驟
        ↓
get_peft_model(model, LoraConfig(...))
        ↓
SFTTrainer(formatting_func, DataCollatorForCompletionOnlyLM)  ← 自動處理 -100 遮罩
        ↓
trainer.train()
        ↓
model.save_pretrained(safe_serialization=True)
        ↓
merge_and_unload()  ← 可選，產生獨立 dense 模型
```

### 關鍵觀念回顧

1. **8-bit vs 4-bit**：8-bit (LLM.int8) 精度損失更小，適合有 8-11 GB VRAM 的場景；4-bit NF4 (QLoRA) 在 < 8 GB VRAM 時是唯一可行的選擇。
2. **`BitsAndBytesConfig` 是正確寫法**：建立設定物件並傳給 `quantization_config=`，讓量化設定可序列化且意圖清晰。
3. **`prepare_model_for_kbit_training()` 順序鐵律**：`load → prepare → get_peft_model`，缺一不可，順序不能錯。
4. **`apply_chat_template()` 確保訓練/推論一致**：訓練時和推論時用同一套模板，是避免分布偏移最直接的方法。
5. **`SFTTrainer` 消除手刻 `-100`**：底層原理（CrossEntropyLoss 忽略 -100）仍值得理解，但日常開發交給 SFTTrainer 即可。

### 練習題

1. 把 `target_modules` 擴展至 `["q_proj", "k_proj", "v_proj", "o_proj"]`，觀察 trainable parameters 增加多少，訓練時間是否明顯改變。
2. 把量化設定改為 4-bit NF4（`load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True`），比較 VRAM 用量與最終推論品質的差異。
3. 在 `SFTConfig` 中設 `packing=True`，觀察訓練速度（steps/sec）的變化，以及 loss 曲線是否有差異。
4. 使用完整資料集（不只前 6000 筆）重新訓練，並將 adapter push 至你的 HF Hub。